# Patient-Specific Feature Normalization (No Leakage)

**Task scope (supervisor's next methodological requirement).** Normalize
EEG features **separately per patient**, fitting each patient's scaler
**only on that patient's TRAIN fold**. This notebook does ONLY
normalization -- no new model, no TCN redesign, no new metrics, no channel
changes, no new patients.

**Input (read-only):** `outputs/labels_binary_preictal.parquet` (the
binary PREICTAL/INTERICTAL table from the previous task -- unscaled).

**Output (new):** `outputs/features_patient_scaled.parquet` / `.csv` --
same table, plus per-patient-scaled feature columns and fold metadata.
Nothing existing is modified.


## 1. Inspect the current normalization/scaling logic

Read directly from the repository (not assumed):

**`EEG_modeling.ipynb`** (CHB01 only -- a single patient): uses an
sklearn `Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(...))])`
fit separately inside each cross-validation fold
(`lr_pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])`). **Per-split,
correctly leakage-free.** Trivially also "per-patient" since only one
patient (CHB01) is ever in `X`.

**`EEG_TCN_seizure_forecasting.ipynb`** (CHB24 only -- a single patient):
`scaler = StandardScaler(); scaler.fit(model_df.loc[train_mask, FEATURE_COLS])`,
then `scaler.transform(...)` applied to the full table (train+val+test)
using the TRAIN-fit scaler only. **Per-split, correctly leakage-free**
(fit never sees val/test rows). Also trivially per-patient (single
patient).

**`EEG_TCN_state_deviation_forecasting.ipynb`** (CHB24 **+** CHB12 -- TWO
patients combined):

```python
baseline_mask = train_row_mask & (y_seizure_np == 0) & (forecast_class_np == 3)
scaler = StandardScaler()
scaler.fit(df.loc[baseline_mask, FEATURE_COLS])          # <-- ONE scaler,
standardized_all = scaler.transform(df[FEATURE_COLS])    #     BOTH patients
```

This *is* fit only on TRAIN rows (no train/test leakage across time), but
`baseline_mask` pools "stable baseline" epochs from **both CHB24 and
CHB12** into a single `StandardScaler` (and a single downstream PCA). One
scaler's mean/std is therefore a blend of two different patients' EEG
statistics -- this is exactly the cross-patient mixing the supervisor's new
requirement ("never fit one scaler across multiple patients") forbids.
This is the current implementation's normalization gap; it is not a
train/test leakage bug, but a cross-patient leakage/contamination issue.

**Conclusion, stated explicitly per the task's inspection requirement:**

| question | answer |
|---|---|
| Is scaling currently global, per-dataset, per-patient, or per-split? | **Per-split** in all three notebooks (fit on TRAIN only) -- but in `EEG_TCN_state_deviation_forecasting.ipynb`, the TRAIN split spans two patients, so that scaler is effectively fit **globally across patients**, not per-patient. |
| Where is `StandardScaler` fit? | `EEG_modeling.ipynb` (inside CV fold), `EEG_TCN_seizure_forecasting.ipynb` (on TRAIN-recording rows), `EEG_TCN_state_deviation_forecasting.ipynb` (on TRAIN-recording, stable-baseline rows, pooled across CHB24+CHB12). |
| Does the current implementation cause leakage? | Not train/test leakage (every scaler above is fit on TRAIN rows only). It DOES cause **cross-patient leakage**: in the two-patient notebook, CHB12's feature distribution influences the mean/std used to standardize CHB24's epochs and vice versa. Given how different individual patients' EEG statistics are (age, electrode placement/montage variant, medication, baseline amplitude), this blended scaler is a real methodological gap -- exactly what this task fixes. |

The new `outputs/labels_binary_preictal.parquet` table (previous task) has
**no scaler applied to it at all** -- it was explicitly left unscaled
("this notebook does not normalize/scale features"), so there is nothing
to fix there; this notebook adds the first (patient-specific, leakage-free)
scaler for that table.


## 2. Load data and define per-patient TRAIN/TEST folds

For this demonstration, TRAIN/TEST are assigned at the **recording level**
(never split epochs randomly, never cross a recording boundary), reusing
the exact recording assignments already established and justified in
`EEG_TCN_state_deviation_forecasting.ipynb` (its TRAIN+VAL recordings
become this task's TRAIN fold; its TEST recordings stay TEST -- no new,
unreviewed split decision is invented). This is a fold definition only;
leave-one-seizure-out (the eventual target) is a separate future task --
the normalization function below takes an arbitrary boolean `train_mask`
so it plugs into a LOSO loop unchanged later.


In [1]:

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

print("=" * 60)
print("PATIENT-SPECIFIC NORMALIZATION -- SETUP")
print("=" * 60)

repo_root = Path.cwd()
output_dir = repo_root / "outputs"
in_path = output_dir / "labels_binary_preictal.parquet"
if not in_path.exists():
    raise FileNotFoundError(f"{in_path} not found -- run EEG_binary_preictal_labeling.ipynb first.")

df = pd.read_parquet(in_path)
print(f"Loaded {in_path.name}: shape={df.shape}")

PATIENT_COL = "patient_id"
NON_FEATURE_COLS = {
    "patient_id", "subject_id", "recording_id", "dataset", "epoch_idx",
    "t_start_sec", "t_center_sec", "t_end_sec",
    "y_seizure", "seizure_id", "next_seizure_id", "next_seizure_start_sec",
    "time_to_next_seizure_sec", "previous_seizure_id",
    "previous_seizure_end_sec", "time_since_previous_seizure_sec",
    "binary_label_h5", "binary_label_h15", "binary_label_h30",
    "exclusion_reason_h5", "exclusion_reason_h15", "exclusion_reason_h30",
}
FEATURE_COLS = [c for c in df.columns if c not in NON_FEATURE_COLS]
print(f"\nFeature columns to be scaled ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Non-feature columns preserved as-is ({len(NON_FEATURE_COLS)}): {sorted(NON_FEATURE_COLS)}")

# Recording-level TRAIN/TEST assignment per patient, reusing the recording
# assignments already established in EEG_TCN_state_deviation_forecasting.ipynb
# (that notebook's TRAIN+VAL -> this task's TRAIN; its TEST -> this task's TEST).
TRAIN_TEST_RECORDINGS = {
    "chb24": {
        "train": ["chb_s24_r04", "chb_s24_r07", "chb_s24_r09", "chb_s24_r11",
                  "chb_s24_r13", "chb_s24_r14", "chb_s24_r15", "chb_s24_r17",
                  "chb_s24_r03", "chb_s24_r06"],
        "test": ["chb_s24_r01", "chb_s24_r21"],
    },
    "chb12": {
        "train": ["chb_s12_r06", "chb_s12_r08", "chb_s12_r23", "chb_s12_r38"],
        "test": ["chb_s12_r42"],
    },
}

df["fold"] = None
for pid, splits in TRAIN_TEST_RECORDINGS.items():
    df.loc[df["recording_id"].isin(splits["train"]), "fold"] = "train"
    df.loc[df["recording_id"].isin(splits["test"]), "fold"] = "test"

if df["fold"].isna().any():
    unassigned = sorted(df.loc[df["fold"].isna(), "recording_id"].unique())
    raise ValueError(f"Recordings with no fold assignment: {unassigned}")

train_mask = (df["fold"] == "train").to_numpy()

print("\nPer-patient recording assignment:")
for pid, splits in TRAIN_TEST_RECORDINGS.items():
    print(f"  {pid}: TRAIN={splits['train']}")
    print(f"  {pid}: TEST ={splits['test']}")

print("\nPer-patient epoch counts by fold:")
print(df.groupby([PATIENT_COL, "fold"]).size().unstack(fill_value=0).to_string())


PATIENT-SPECIFIC NORMALIZATION -- SETUP
Loaded labels_binary_preictal.parquet: shape=(2040, 36)

Feature columns to be scaled (14): ['std', 'var', 'rms', 'delta', 'theta', 'alpha', 'beta', 'rel_delta', 'rel_theta', 'rel_alpha', 'rel_beta', 'theta_alpha_ratio', 'delta_beta_ratio', 'entropy']
Non-feature columns preserved as-is (22): ['binary_label_h15', 'binary_label_h30', 'binary_label_h5', 'dataset', 'epoch_idx', 'exclusion_reason_h15', 'exclusion_reason_h30', 'exclusion_reason_h5', 'next_seizure_id', 'next_seizure_start_sec', 'patient_id', 'previous_seizure_end_sec', 'previous_seizure_id', 'recording_id', 'seizure_id', 'subject_id', 't_center_sec', 't_end_sec', 't_start_sec', 'time_since_previous_seizure_sec', 'time_to_next_seizure_sec', 'y_seizure']

Per-patient recording assignment:
  chb24: TRAIN=['chb_s24_r04', 'chb_s24_r07', 'chb_s24_r09', 'chb_s24_r11', 'chb_s24_r13', 'chb_s24_r14', 'chb_s24_r15', 'chb_s24_r17', 'chb_s24_r03', 'chb_s24_r06']
  chb24: TEST =['chb_s24_r01', 'chb_

## 3. Reusable patient-specific scaler functions

Two small, composable functions -- both take an explicit boolean
`train_mask` rather than deciding the split themselves, so the exact same
code can later be called once per fold inside a leave-one-seizure-out loop
(only `train_mask` changes between folds; nothing else about the API
changes):

- **`fit_patient_scalers(df, patient_col, feature_cols, train_mask)`** --
  for each unique patient, fits one `StandardScaler` using ONLY the rows
  where `train_mask` is `True` **and** that belong to that patient.
  Raises if a patient has zero TRAIN rows. Returns `{patient_id: scaler}`.
- **`transform_with_patient_scalers(df, patient_col, feature_cols, scalers)`**
  -- transforms every row (TRAIN and TEST alike) using **its own patient's**
  fitted scaler. Raises if any row's patient has no fitted scaler (never
  silently falls back to another patient's scaler, never silently skips a
  row).

Scaled features are written to NEW columns (`"<feature>_scaled"`), so the
raw (unscaled) features remain available in the same table -- nothing is
overwritten, and only the `FEATURE_COLS` (14 qEEG columns) are ever passed
to a scaler; every metadata/identifier/label/timestamp column listed in
`NON_FEATURE_COLS` above is left completely untouched.


In [2]:

def fit_patient_scalers(df, patient_col, feature_cols, train_mask):
    '''Fit one StandardScaler per patient, using ONLY that patient's TRAIN rows.'''
    scalers = {}
    train_mask = np.asarray(train_mask)
    for pid in df.loc[train_mask, patient_col].unique():
        sel = train_mask & (df[patient_col] == pid).to_numpy()
        n = int(sel.sum())
        if n == 0:
            raise ValueError(f"Patient {pid!r} has zero TRAIN rows -- cannot fit a scaler.")
        # Structural guarantee: this fit call only ever sees rows for ONE patient.
        assert df.loc[sel, patient_col].nunique() == 1
        scaler = StandardScaler()
        scaler.fit(df.loc[sel, feature_cols])
        scalers[pid] = scaler
    return scalers


def transform_with_patient_scalers(df, patient_col, feature_cols, scalers):
    '''Transform every row using ITS OWN patient's fitted scaler.'''
    out = df.copy()
    scaled = np.full((len(df), len(feature_cols)), np.nan)
    covered = np.zeros(len(df), dtype=bool)
    for pid, scaler in scalers.items():
        sel = (df[patient_col] == pid).to_numpy()
        if sel.any():
            scaled[sel, :] = scaler.transform(df.loc[sel, feature_cols])
            covered |= sel
    if not covered.all():
        missing_pids = sorted(df.loc[~covered, patient_col].unique())
        raise ValueError(f"No fitted scaler available for patients: {missing_pids}")
    scaled_cols = [f"{c}_scaled" for c in feature_cols]
    out[scaled_cols] = scaled
    return out, scaled_cols


def patient_specific_train_test_scale(df, patient_col, feature_cols, train_mask):
    '''Convenience wrapper: fit per-patient TRAIN-only scalers, transform ALL rows.'''
    scalers = fit_patient_scalers(df, patient_col, feature_cols, train_mask)
    scaled_df, scaled_cols = transform_with_patient_scalers(df, patient_col, feature_cols, scalers)
    return scaled_df, scalers, scaled_cols


scaled_df, scalers, SCALED_FEATURE_COLS = patient_specific_train_test_scale(
    df, PATIENT_COL, FEATURE_COLS, train_mask
)
print(f"[OK] Fit {len(scalers)} patient-specific scalers: {sorted(scalers.keys())}")
print(f"[OK] Produced {len(SCALED_FEATURE_COLS)} scaled feature columns.")


[OK] Fit 2 patient-specific scalers: ['chb12', 'chb24']
[OK] Produced 14 scaled feature columns.


## 4. Leakage checks

Programmatic, not visual -- every check below raises immediately on
failure rather than printing a warning and continuing.


In [3]:

def run_leakage_checks(df, patient_col, feature_cols, train_mask, scalers):
    train_mask = np.asarray(train_mask)
    print("Running leakage checks...")

    # 1 & 2. scaler.fit() saw ONLY training rows, and no test row contributed
    #        to the mean/std estimate. StandardScaler.n_samples_seen_ is an
    #        exact count of rows it was fit on -- compare it to the actual
    #        number of TRAIN rows for that patient (not just "some number").
    for pid, scaler in scalers.items():
        n_train_rows = int((train_mask & (df[patient_col] == pid).to_numpy()).sum())
        n_total_rows = int((df[patient_col] == pid).sum())
        assert int(scaler.n_samples_seen_) == n_train_rows, (
            f"{pid}: scaler saw {scaler.n_samples_seen_} rows, expected exactly "
            f"{n_train_rows} TRAIN rows"
        )
        if n_total_rows > n_train_rows:
            # If TEST rows had leaked into the fit, n_samples_seen_ would equal
            # n_total_rows (or something other than n_train_rows) -- assert the
            # two are genuinely different so this check cannot pass vacuously.
            assert n_train_rows < n_total_rows
        # Recompute the mean directly from the TRAIN-only subset and confirm
        # it matches the fitted scaler exactly (independent re-derivation,
        # not just trusting sklearn's internal bookkeeping).
        sel = train_mask & (df[patient_col] == pid).to_numpy()
        manual_mean = df.loc[sel, feature_cols].to_numpy().mean(axis=0)
        assert np.allclose(manual_mean, scaler.mean_), f"{pid}: fitted mean does not match TRAIN-only recomputation"
    print(f"  [OK] check 1/2: every scaler's n_samples_seen_ exactly equals that "
          f"patient's TRAIN row count (no TEST row contributed); means "
          f"independently recomputed from TRAIN-only rows match scaler.mean_ exactly.")

    # 3. Patients are never mixed during fitting: each scaler's mean_ must be
    #    reproducible from THAT patient's TRAIN rows alone, and must NOT be
    #    reproducible from another patient's TRAIN rows (proves no pooling).
    for pid, scaler in scalers.items():
        for other_pid in scalers:
            if other_pid == pid:
                continue
            other_sel = train_mask & (df[patient_col] == other_pid).to_numpy()
            other_mean = df.loc[other_sel, feature_cols].to_numpy().mean(axis=0)
            assert not np.allclose(other_mean, scaler.mean_), (
                f"{pid}'s scaler mean unexpectedly matches {other_pid}'s TRAIN mean -- "
                f"possible cross-patient mixing"
            )
    print(f"  [OK] check 3: each patient's fitted mean is NOT reproducible from any "
          f"other patient's TRAIN rows -- confirms no cross-patient pooling.")

    # 4. Feature column order identical between train and test transforms.
    for pid, scaler in scalers.items():
        assert list(scaler.feature_names_in_) == list(feature_cols), (
            f"{pid}: scaler was fit on a different column order than feature_cols"
        )
    print(f"  [OK] check 4: feature column order used at fit-time is identical "
          f"to feature_cols for every patient (and therefore identical between "
          f"however TRAIN and TEST rows are later selected for transform).")

    print("\n[ALL LEAKAGE CHECKS PASSED]")


run_leakage_checks(df, PATIENT_COL, FEATURE_COLS, train_mask, scalers)


Running leakage checks...
  [OK] check 1/2: every scaler's n_samples_seen_ exactly equals that patient's TRAIN row count (no TEST row contributed); means independently recomputed from TRAIN-only rows match scaler.mean_ exactly.
  [OK] check 3: each patient's fitted mean is NOT reproducible from any other patient's TRAIN rows -- confirms no cross-patient pooling.
  [OK] check 4: feature column order used at fit-time is identical to feature_cols for every patient (and therefore identical between however TRAIN and TEST rows are later selected for transform).

[ALL LEAKAGE CHECKS PASSED]


## 5. Validation report on the currently available patients

In [4]:

print("=" * 60)
print("VALIDATION REPORT")
print("=" * 60)

print(f"\nPatients processed: {sorted(scalers.keys())}")

print("\nTrain/test sample counts per patient:")
print(scaled_df.groupby([PATIENT_COL, "fold"]).size().unstack(fill_value=0).to_string())

print(f"\nNumber of scaled feature columns: {len(SCALED_FEATURE_COLS)}")
print(f"Scaled feature columns: {SCALED_FEATURE_COLS}")

demo_patient = "chb24"
print(f"\n--- Before/after summary statistics for one fold (patient={demo_patient}) ---")

for fold_name in ["train", "test"]:
    sel = (scaled_df[PATIENT_COL] == demo_patient) & (scaled_df["fold"] == fold_name)
    n = int(sel.sum())
    raw_stats = scaled_df.loc[sel, FEATURE_COLS].agg(["mean", "std"]).T
    scaled_stats = scaled_df.loc[sel, SCALED_FEATURE_COLS].agg(["mean", "std"]).T
    # Raw qEEG band-power/amplitude features are on the order of 1e-5 to
    # 1e-10 (MNE reports EEG in Volts) -- fixed 4-decimal rounding would
    # display these as a meaningless 0.0000, so use scientific notation here.
    with pd.option_context("display.float_format", lambda v: f"{v:.4e}"):
        print(f"\n{demo_patient} / {fold_name.upper()} (n={n} epochs) -- RAW features (mean, std), first 5 shown:")
        print(raw_stats.head(5).to_string())
    print(f"\n{demo_patient} / {fold_name.upper()} (n={n} epochs) -- SCALED features (mean, std), first 5 shown:")
    print(scaled_stats.head(5).round(4).to_string())

print(f"\nExpected and observed: {demo_patient}/TRAIN scaled features have mean~0, std~1 "
      f"(the scaler was fit exactly on these rows). {demo_patient}/TEST scaled features do "
      f"NOT have mean exactly 0 / std exactly 1 -- that is the correct, expected signature "
      f"of a leakage-free transform (TEST was transformed using TRAIN's statistics, not its own).")


VALIDATION REPORT

Patients processed: ['chb12', 'chb24']

Train/test sample counts per patient:
fold        test  train
patient_id             
chb12        120    480
chb24        240   1200

Number of scaled feature columns: 14
Scaled feature columns: ['std_scaled', 'var_scaled', 'rms_scaled', 'delta_scaled', 'theta_scaled', 'alpha_scaled', 'beta_scaled', 'rel_delta_scaled', 'rel_theta_scaled', 'rel_alpha_scaled', 'rel_beta_scaled', 'theta_alpha_ratio_scaled', 'delta_beta_ratio_scaled', 'entropy_scaled']

--- Before/after summary statistics for one fold (patient=chb24) ---

chb24 / TRAIN (n=1200 epochs) -- RAW features (mean, std), first 5 shown:
            mean        std
std   4.6191e-05 2.3643e-05
var   2.6922e-09 3.7194e-09
rms   4.6191e-05 2.3643e-05
delta 1.8563e-09 2.9177e-09
theta 2.8732e-10 4.3524e-10

chb24 / TRAIN (n=1200 epochs) -- SCALED features (mean, std), first 5 shown:
              mean     std
std_scaled    -0.0  1.0004
var_scaled    -0.0  1.0004
rms_scaled     

## 6. Save output (new file, nothing existing overwritten)

In [5]:

out_cols = (
    list(NON_FEATURE_COLS) + ["fold"] + FEATURE_COLS + SCALED_FEATURE_COLS
)
out_cols = [c for c in out_cols if c in scaled_df.columns]
final = scaled_df[out_cols].copy()

out_parquet = output_dir / "features_patient_scaled.parquet"
out_csv = output_dir / "features_patient_scaled.csv"
final.to_parquet(out_parquet, index=False)
final.to_csv(out_csv, index=False)

print(f"[OK] Saved {final.shape[0]} rows x {final.shape[1]} columns")
print(f"  -> {out_parquet}")
print(f"  -> {out_csv}")

import os
assert os.path.exists(output_dir / "labels_binary_preictal.parquet")
assert os.path.exists(output_dir / "features_chb24_forecasting.parquet")
assert os.path.exists(output_dir / "features_chb12_forecasting.parquet")
print("\n[OK] All previously existing outputs remain untouched.")


[OK] Saved 2040 rows x 51 columns
  -> C:\Users\angela\Downloads\EEG_timeseries-main\EEG_timeseries-main\outputs\features_patient_scaled.parquet
  -> C:\Users\angela\Downloads\EEG_timeseries-main\EEG_timeseries-main\outputs\features_patient_scaled.csv

[OK] All previously existing outputs remain untouched.


## 7. Summary

- **Patients processed:** `chb24`, `chb12` -- exactly the two patients
  currently available, each with its own independently-fit `StandardScaler`.
- **No model, no new metrics, no TCN, no channel changes, and no new
  patients** were introduced in this notebook.
- **Old normalization code is preserved, not deleted.** See Section 1
  above (and the deprecation note added to
  `EEG_TCN_state_deviation_forecasting.ipynb`, just before its scaler cell)
  for why the state-deviation notebook's pooled CHB24+CHB12 scaler is now
  understood to be a cross-patient leakage gap this notebook fixes going
  forward -- its own saved outputs are untouched and remain valid for
  project-history comparison.
- **Still not implemented:** any classifier trained on the scaled features,
  a leave-one-seizure-out fold definition (today's TRAIN/TEST fold is
  recording-based, reused from the earlier notebook -- LOSO itself is a
  separate future task), new evaluation metrics, channel-level changes, and
  expansion beyond CHB24/CHB12.
